In [1]:
from rdkit import Chem
import pandas as pd
from ml_enhance import QuantumFPFileLoader
import numpy as np
from ml_enhance import parallelize

In [28]:
# loader = QuantumFPFileLoader("../data/QM9/output")
loader = QuantumFPFileLoader("../data/QFP_rerun/output")
files = loader.list_output_files()

In [29]:
wanted_properties = ["original_smiles", "output_smiles", "energy", "molecular_dipole_norm", "molecular_polarizability_mean", "homo_lumo_gap", "zero_point_energy", "atomization_energy", "xyz"]

In [30]:
for df in loader.stream_conformer_dataframe(files[0], include_coords=True):
    display(df["original_smiles"])

0    [O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...
Name: original_smiles, dtype: string

In [31]:
def get_min_series(file):
    for df in loader.stream_conformer_dataframe(file, include_coords=True):
        min_series = df.loc[df["energy"].argmin(), wanted_properties]
        return min_series

In [32]:
min_seriess = parallelize(get_min_series, files)

min_df = pd.DataFrame(min_seriess).reset_index()

100%|██████████| 8833/8833 [29:09<00:00,  5.05it/s]  


In [33]:
min_df.head()

,index,original_smiles,output_smiles,energy,molecular_dipole_norm,molecular_polarizability_mean,homo_lumo_gap,zero_point_energy,atomization_energy,xyz
0,0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,-34.129004,1.514291,-99.743511,0.089533,92.688240,5.239994,"[[1, -3.215995401162942, -0.6117693337828871, ..."
1,0,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,-26.123298,0.965419,-95.249802,0.105970,60.507563,3.330826,"[[1, -2.987291144734709, -0.15605563901242914,..."
2,13,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,-45.124599,0.495893,-97.623141,0.156383,174.625936,7.142925,"[[1, -3.4293012719398734, -1.9108930927712622,..."
3,2,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,-39.730889,0.480588,-80.548992,0.127745,123.883937,5.733924,"[[1, -1.1540116803690164, -2.468296503291047, ..."
4,5,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,-72.428696,0.629000,-229.679947,0.064549,141.082325,8.520828,"[[1, -7.082350627805967, 0.9674638834465076, -..."


In [34]:
min_df.to_json("aqsol_mols.json", index=False)